# MetaPy 5.0: intelligent meta-analysis assistant (Google Colab)

**Workflow:** install → upload the bundle → test → launch the app → upload your data → follow the 5 tabs.

In the app MetaPy will:
1. read your CSV or Excel file and list every analysis your data can support
2. derive missing statistics with its derivation brain (167 cited formulas): exact formulas automatically, published approximations listed for your approval, assumptions only when you switch them on
3. ask you for more data **only** when nothing can be derived, telling you exactly which statistics would do
4. run the analysis, let you review it and add analyses, then build tables, figures (including PRISMA 2020) and export to Word, Excel, PDF, HTML, CSV or images

Version 5 adds three-level and robust-variance (CHE-RVE) models for several effects per study, network meta-analysis, selection models, PET-PEESE, Peters' test, Q-profile intervals for τ²/I², Bayesian random effects and multiple meta-regression with permutation tests (Step 3 of the app, *Additional analyses* and the *Network meta-analysis* panel).

Your original file is never modified. Use de-identified, aggregate data only.

## Step 1: Install the required packages

In [ ]:
%pip -q install "pandas>=2.0" "numpy>=1.24" "scipy>=1.10" "matplotlib>=3.7" "openpyxl>=3.1" "python-docx>=1.1" "reportlab>=4.0" pillow "gradio>=5,<7"
print("Environment ready")

## Step 2: Find the MetaPy bundle in `/content`

Upload the MetaPy bundle zip (e.g. `MetaPy_v5_0_0_Bundle.zip`) once through the **Files** panel (folder icon on the left → upload icon), into `/content`. You can also put it in Google Drive (see the note in the cell). This cell does **not** open an upload dialog; it just finds the zip.

In [ ]:
from pathlib import Path

# Optional: keep the zip in Google Drive so you never re-upload it after a runtime reset.
# Uncomment the next two lines and put the zip in MyDrive/MetaPy/
# from google.colab import drive; drive.mount("/content/drive")
SEARCH_DIRS = [Path("/content"), Path("/content/drive/MyDrive/MetaPy"), Path("/content/drive/MyDrive")]

candidates = []
for d in SEARCH_DIRS:
    if d.is_dir():
        candidates += [p for p in d.glob("*.zip") if p.name.lower().startswith("metapy")]
if not candidates:
    raise FileNotFoundError(
        "No MetaPy zip found. Upload the MetaPy bundle zip to /content with the Files panel "
        "(folder icon on the left), then run this cell again.")
ZIP_PATH = max(candidates, key=lambda p: p.stat().st_mtime)   # newest MetaPy zip
print("Using:", ZIP_PATH, f"({ZIP_PATH.stat().st_size // 1024} KB)")

## Step 3: Extract safely and add MetaPy to Python

In [ ]:
import os, sys, shutil, zipfile, importlib
from pathlib import Path

ROOT = Path("/content/MetaPy")
TMP = Path("/content/_metapy_extract")
# 1) remove any previously extracted version so old files cannot linger
for d in (ROOT, TMP):
    if d.exists():
        shutil.rmtree(d)
TMP.mkdir(parents=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    for item in z.infolist():
        target = (TMP / item.filename).resolve()
        if not target.is_relative_to(TMP.resolve()):
            raise ValueError(f"Unsafe path in ZIP: {item.filename}")
    z.extractall(TMP)

# 2) find the folder that contains the metapy package, whatever the zip's folder layout
hits = sorted(TMP.rglob("metapy/__init__.py"), key=lambda p: len(p.parts))
hits = [p for p in hits if "legacy" not in str(p).lower() and "__MACOSX" not in str(p)]
if not hits:
    raise FileNotFoundError(f"No metapy/__init__.py inside {ZIP_PATH.name}. Is this the MetaPy bundle zip?")
shutil.move(str(hits[0].parent.parent), str(ROOT))      # the project folder (contains metapy/, tests/, README.md)
shutil.rmtree(TMP, ignore_errors=True)

# 3) forget any MetaPy modules already loaded in this runtime (otherwise old code keeps running)
for name in [m for m in sys.modules if m == "metapy" or m.startswith("metapy.")]:
    del sys.modules[name]
sys.path = [p for p in sys.path if "MetaPy" not in p]
sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
import metapy
print("MetaPy", metapy.__version__, "loaded from", Path(metapy.__file__).parent)
print("Tests folder:", ROOT / "tests", "found" if (ROOT / "tests").is_dir() else "MISSING")
print("If this is not the version you uploaded: Runtime > Restart session, then run from Step 1 again.")

## Step 4: Run the tests
Includes checks against metafor reference values for the BCG vaccine dataset.

In [ ]:
!cd /content/MetaPy && python -m unittest discover -s tests -v

## Step 5: Launch the MetaPy app
A link appears below. To avoid a public share link, set `share=False` (the app then opens inside Colab only).

To try MetaPy first, download the synthetic demo in Step 6 and upload it in the app.

In [ ]:
import metapy
print("Launching MetaPy", metapy.__version__)   # check this matches the bundle you uploaded
from metapy.app import launch
import gradio as gr
gr.close_all()
launch(share=True, debug=True)

## Step 6 (optional): Download the synthetic demo file
It is **synthetic test data**: messy on purpose (medians, text cells, t values, p<0.05, unlabelled SMDs, a multi-arm study, percentages). Not real studies.

In [ ]:
from google.colab import files
files.download("/content/MetaPy/examples/MetaPy_SYNTHETIC_demo.xlsx")

## Step 7 (optional): Use MetaPy from code instead of the app

In [ ]:
from metapy import MetaPySession
s = MetaPySession()
scan = s.load("/content/MetaPy/examples/MetaPy_SYNTHETIC_demo.xlsx")
for p in scan.plans:
    print(p["summary"])

pid = scan.plans[0]["plan_id"]
s.select_plans([pid])
prep = s.prepare(pid)
display(prep.actions)          # what MetaPy calculated
for q in prep.questions:       # decisions MetaPy needs
    print("?", q["question"], [o[0] for o in q["options"]])
display(prep.requests)         # data still needed

## Notes
- Every derived value is listed with formula, inputs and citation in the derivation log (Step 2 of the app, and in the exported report).
- MetaPy checks internal consistency only; it cannot verify values against the original papers.
- Validate key results against an independent implementation before publication.
- The full rule reference is in `docs/DERIVATION_RULES.md`.